# A dataframe from a CSV file

`Csv` reads a file whose columns the caller declares (no inference:
every kind is said), `Frame` holds the columns and knows the verbs --
`GroupBy`, `Describe`, `Filter`, `Join` -- each held to polars. The
first program writes a small file so the notebook stands alone.

In [ ]:
MODULE WriteObs ;

IMPORT Io ;

CONST Nl = 0AC ;

BEGIN
  Io.WriteFile ('obs.csv',
    'station,year,co2' + Nl
    + 'HTM,2019,410.1' + Nl + 'HTM,2020,412.6' + Nl + 'HTM,2021,414.9' + Nl + 'HTM,2022,417.3' + Nl
    + 'SMR,2019,409.4' + Nl + 'SMR,2020,411.8' + Nl + 'SMR,2021,414.2' + Nl + 'SMR,2022,416.5' + Nl
    + 'ZEP,2019,408.2' + Nl + 'ZEP,2020,410.9' + Nl + 'ZEP,2021,413.0' + Nl + 'ZEP,2022,415.6' + Nl) ;
  Io.WriteLine ('obs.csv written: 3 stations, 4 years')
EXCEPT
| ValueRange :
    Io.ErrLine ('a value out of range')
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
END WriteObs.

A state cell opens the file, declares the three columns and keeps the
frame, so that every cell below reads `Obs.f`.

In [ ]:
STATEFUL DEFINITION MODULE Obs ;

IMPORT Frame ;

VAR f : PTR Frame.Fr ;

END Obs.

IMPLEMENTATION MODULE Obs ;

IMPORT Csv ;
IMPORT Faults ;
IMPORT Frame ;
IMPORT Io ;

VAR
  pool : POOL ;
  t : PTR Csv.Table IN pool ;
  opt : Csv.Options ;

BEGIN
  opt := Csv.Defaults () ;
  t := Csv.Open (pool, 'obs.csv', opt) ;
  Csv.SetText (t, 0) ;
  Csv.SetInt (t, 1) ;
  Csv.SetReal64 (t, 2) ;
  Csv.Parse (pool, t) ;
  f := Frame.FromCsv (pool, t)
EXCEPT
| Csv.ParseError (what, line, col) :
    Io.ErrLine ('obs.csv: ' + what)
| Csv.RangeError (row, col) :
    Io.ErrLine ('obs.csv: a value out of range')
| Frame.Duplicate (name) :
    Io.ErrLine ('a column twice: ' + name)
| Faults.SizeError (got, want) :
    Io.ErrLine ('the sizes disagree')
| Io.IOError (path) :
    Io.ErrLine ('cannot read ' + path)
| ValueRange :
    Io.ErrLine ('a value out of range')
| IndexError :
    Io.ErrLine ('out of range')
END Obs.

In [ ]:
Obs.f

In [ ]:
Frame.Rows (Obs.f)

`GroupBy` reduces every other column over the rows of each key, one
`How` per column -- the key's entry is ignored, an integer column
takes `HowLo`/`HowHi`/`HowFirst`/`HowLast`, a real one `HowMean` or
`HowSum` as well; `Describe` is the nine statistics of every numeric
column. `NbShow.TableText` is the text a frame shows as.

In [ ]:
MODULE Groups ;

IMPORT Faults ;
IMPORT Frame ;
IMPORT Io ;
IMPORT NbCells ;
IMPORT NbShow ;
IMPORT Obs ;
IMPORT Plot ;

VAR
  pool : POOL ;
  how : SLICE OF Frame.How ;
  g, d : PTR Frame.Fr IN pool ;
  at, v : SLICE OF F64 ;
  i : I64 ;

BEGIN
  how := NEW (pool, Frame.How, 3) ;
  how[0] := Frame.HowFirst () ;      (* the key's entry is ignored *)
  how[1] := Frame.HowLo () ;         (* year: the first of the group *)
  how[2] := Frame.HowMean () ;       (* co2: the mean *)
  g := Frame.GroupBy (pool, Obs.f, 'station', how, 1) ;
  Io.WriteLine ('mean by station:') ;
  Io.WriteLine (NbShow.TableText (g)) ;
  d := Frame.Describe (pool, Obs.f) ;
  Io.WriteLine ('describe:') ;
  Io.WriteLine (NbShow.TableText (d)) ;
  v := Frame.ColF64 (g, 'co2') ;
  at := NEW (pool, F64, LEN (v)) ;
  FOR i := 0 TO LEN (v) - 1 DO at[i] := F64 (i + 1) END ;
  Plot.ClearFigure () ;
  Plot.AddBars (at, v, 0, 'mean co2 by station: HTM, SMR, ZEP') ;
  Io.WriteFile ('bystation.svg', Plot.Render ('mean CO2 2019-2022', 'station', 'ppm')) ;
  NbCells.PutFrame (g, 'bystation')
EXCEPT
| Frame.Unknown (name) :
    Io.ErrLine ('no column ' + name)
| Frame.WrongType (name) :
    Io.ErrLine ('wrong type: ' + name)
| Frame.Duplicate (name) :
    Io.ErrLine ('a column twice: ' + name)
| Faults.SizeError (got, want) :
    Io.ErrLine ('the sizes disagree')
| Faults.BadArg (what) :
    Io.ErrLine ('bad argument: ' + what)
| Io.IOError (path) :
    Io.ErrLine ('cannot write ' + path)
| ValueRange :
    Io.ErrLine ('a value out of range')
| Overflow :
    Io.ErrLine ('overflow')
| IndexError :
    Io.ErrLine ('out of range')
END Groups.

A filter is a mask: `Arrays` makes one from a column, `Frame.Filter`
keeps the rows where it is true. Strings in the answer are views of
the source frame's.

In [ ]:
MODULE High ;

IMPORT Arrays ;
IMPORT Faults ;
IMPORT Frame ;
IMPORT Io ;
IMPORT NbShow ;
IMPORT Obs ;

VAR
  pool : POOL ;
  h : PTR Frame.Fr IN pool ;

BEGIN
  h := Frame.Filter (pool, Obs.f, Arrays.Greater (Frame.ColF64 (Obs.f, 'co2'), 414.0)) ;
  Io.WriteLine ('rows above 414 ppm:') ;
  Io.WriteLine (NbShow.TableText (h))
EXCEPT
| Frame.Unknown (name) :
    Io.ErrLine ('no column ' + name)
| Frame.WrongType (name) :
    Io.ErrLine ('wrong type: ' + name)
| Frame.Duplicate (name) :
    Io.ErrLine ('a column twice: ' + name)
| Faults.SizeError (got, want) :
    Io.ErrLine ('the sizes disagree')
| IndexError :
    Io.ErrLine ('out of range')
END High.

`bystation` is now `04-dataframes.m9data/bystation.frame.parquet`
beside the notebook: `pd.read_parquet` reads it, and a later M9 cell
gets it back with `NbCells.GetFrame`.